# AI Fraud Detection and Adversarial Robustness
## PHASE 1 — Fraud Detection
Run the following Phase 1 cells in order. Later phases remain placeholders.

### Cell 1 — Environment / Imports

In [ ]:
from pathlib import Path
import json
import os
project_candidates = [Path.cwd(), Path('/content/AI_Fraud_Adversarial')]
PROJECT_ROOT = next((p for p in project_candidates if (p / 'requirements-colab.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Clone the GitHub repository to /content/AI_Fraud_Adversarial, then rerun this cell.')
os.chdir(PROJECT_ROOT)
%pip install -q -r requirements-colab.txt
import joblib
import matplotlib.pyplot as plt
import pandas as pd


### Cell 2 — Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3 — Configuration

In [ ]:
DATASET_PATH = Path('/content/drive/MyDrive/AI_Fraud_Adversarial/data/paysim.csv')
DEVELOPMENT_MODE = True  # Set False only for the final full-data run.
RUN_MODE = 'DEVELOPMENT_MODE' if DEVELOPMENT_MODE else 'FULL_MODE'
RANDOM_SEED = 42
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_Fraud_Adversarial/outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase1'
MODELS_DIR = OUTPUT_DIR / 'models'
METRICS_DIR = OUTPUT_DIR / 'metrics'
for directory in (FIGURES_DIR, MODELS_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'run_mode': RUN_MODE, 'dataset': str(DATASET_PATH), 'outputs': str(OUTPUT_DIR)})

### Cell 4 — Dataset Check

In [ ]:
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'PaySim CSV not found: {DATASET_PATH}')
print(f'PaySim found: {DATASET_PATH} ({DATASET_PATH.stat().st_size / 1e9:.2f} GB)')

### Cell 5 — Load PaySim

In [ ]:
from src.data_loader import load_paysim
data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
print(f'Loaded {len(data):,} rows in {RUN_MODE}.')

### Cell 6 — Inspect Dataset

In [ ]:
from src.data_loader import summarize_paysim
summary = summarize_paysim(data)
print('Shape:', summary['shape'])
print('Columns:', summary['columns'])
print('Missing values:', summary['missing_values'])
print('Transaction types:', summary['transaction_types'])
print('Class counts:', summary['class_counts'])
print(f"Fraud percentage: {summary['fraud_percentage']:.6f}%")

### Cell 7 — Preprocess and Engineer Features

In [ ]:
from src.preprocessing import prepare_features_and_target
X, y = prepare_features_and_target(data)
print('Raw model inputs:', X.columns.tolist())
print('Target counts:', y.value_counts().sort_index().to_dict())

### Cell 8 — Train/Test Split

In [ ]:
from src.preprocessing import fit_transform_train_test
from src.train import stratified_train_test_split
X_train_raw, X_test_raw, y_train, y_test = stratified_train_test_split(
    X, y, random_state=RANDOM_SEED
)
X_train, X_test, preprocessor = fit_transform_train_test(X_train_raw, X_test_raw)
feature_names = X_train.columns.tolist()
print('Training counts:', y_train.value_counts().sort_index().to_dict())
print('Untouched test counts:', y_test.value_counts().sort_index().to_dict())
print('Encoded features:', feature_names)

### Cell 9 — SMOTE

In [ ]:
from src.config import SMOTE_SAMPLING_STRATEGY
from src.preprocessing import resample_training_data
print('Before SMOTE:', y_train.value_counts().sort_index().to_dict())
X_train_smote, y_train_smote = resample_training_data(
    X_train, y_train, random_state=RANDOM_SEED,
    sampling_strategy=SMOTE_SAMPLING_STRATEGY
)
print('After SMOTE:', pd.Series(y_train_smote).value_counts().sort_index().to_dict())
print('Test data was not passed to SMOTE.')

### Cell 10 — Train Baseline Model

In [ ]:
from src.train import train_baseline_model
baseline_model = train_baseline_model(X_train_smote, y_train_smote)
print('Baseline XGBoost training complete.')

### Cell 11 — Evaluate Baseline

In [ ]:
from src.evaluate import evaluate_binary_classifier
from src.visualization import (
    plot_confusion_matrix, plot_original_class_distribution,
    plot_precision_recall, plot_smote_distributions,
)
evaluation = evaluate_binary_classifier(baseline_model, X_test, y_test)
print(json.dumps(evaluation['metrics'], indent=2))
print('Fraud Recall (primary concern):', evaluation['metrics']['recall'])
print(pd.DataFrame(evaluation['classification_report']).transpose())
plot_original_class_distribution(y, FIGURES_DIR)
plot_smote_distributions(y_train, y_train_smote, FIGURES_DIR)
plot_confusion_matrix(evaluation['confusion_matrix'], FIGURES_DIR)
plot_precision_recall(evaluation, FIGURES_DIR)
plt.show()

### Cell 12 — Save Artifacts

In [ ]:
from src.evaluate import save_metrics
baseline_model_path = MODELS_DIR / 'baseline_model.joblib'
feature_names_path = MODELS_DIR / 'feature_names.joblib'
preprocessor_path = MODELS_DIR / 'baseline_preprocessor.joblib'
metrics_path = METRICS_DIR / 'phase1_baseline_metrics.json'
joblib.dump(baseline_model, baseline_model_path)
joblib.dump(feature_names, feature_names_path)
joblib.dump(preprocessor, preprocessor_path)
save_metrics(evaluation, metrics_path)
print('Saved:', baseline_model_path, feature_names_path, preprocessor_path, metrics_path, sep='\n- ')

# PHASE 2 — SHAP Explainability
Not implemented.

# PHASE 3 — Adversarial Attack
Not implemented.

# PHASE 4 — Multiple Attack Comparison
Not implemented.

# PHASE 5 — Model Hardening
Not implemented.

# PHASE 6 — Streamlit Dashboard
Not implemented.

# PHASE 7 — Real-Time Transaction Simulation
Not implemented.

# PHASE 8 — Concept Drift
Not implemented.

# FINAL — Results and Conclusions
Complete after all phases have actual execution results.